# Disaster Data Preprocessing

## Objective

Build an India-scoped disaster response dataset by combining:

- USGS earthquake data
- OpenMeteo weather data
- India WorldPop population raster output from `WorldPop_API.ipynb`
- Event-local OpenStreetMap infrastructure counts from `OSM_API.ipynb`

## Core Logic

- Keep only disaster events covered by the selected India WorldPop raster.
- Sample population at each disaster coordinate.
- Merge infrastructure counts by `SourceEventID`, so counts are local to each disaster event and search radius.
- Save `datasets/final_disaster_dataset.csv`.


In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import rasterio
from rasterio.windows import Window


current_dir = Path.cwd()
PROJECT_ROOT = current_dir.parent if current_dir.name == "notebooks" else current_dir
DATA_DIR = PROJECT_ROOT / "datasets"

EARTHQUAKE_PATH = DATA_DIR / "earthquake_data.csv"
WEATHER_PATH = DATA_DIR / "weather_data.csv"
POPULATION_RASTER_PATH = DATA_DIR / "population_data.tif"
POPULATION_SELECTED_METADATA_PATH = DATA_DIR / "population_selected_metadata.json"
#EVENT_INFRASTRUCTURE_PATH = DATA_DIR / "event_infrastructure_summary.csv"
FINAL_DATASET_PATH = DATA_DIR / "final_disaster_dataset.csv"
GDACS_DATA_PATH = (DATA_DIR /"gdacs_data.csv")
EARTHQUAKE_DATA_PATH = (DATA_DIR /"earthquake_data.csv")

#WEATHER_DATA_PATH = (DATA_DIR /"weather_data.csv")
WEATHER_DATA_PATH = (DATA_DIR /"weather_useful_events.csv")

INFRASTRUCTURE_DATA_PATH = (DATA_DIR /"event_infrastructure_summary.csv")
COMBINED_DATA_PATH = (DATA_DIR /"combined_disasters.csv")

FILTER_TO_POPULATION_COVERAGE = True
DEFAULT_INFRASTRUCTURE_RADIUS_KM = 25.0
REQUIRE_EVENT_INFRASTRUCTURE = False

INFRASTRUCTURE_COLUMNS = [
    "Hospitals",
    "Buildings",
    "Roads",
    "Bridges",
    "FireStations",
    "Clinics",
    "Schools",
]

print(f"Project root: {PROJECT_ROOT}")
print(f"Datasets folder: {DATA_DIR}")


Project root: C:\Users\aashi\PycharmProjects\AI-Based Disaster Response Management System for Resource Allocation and Relief Coordination
Datasets folder: C:\Users\aashi\PycharmProjects\AI-Based Disaster Response Management System for Resource Allocation and Relief Coordination\datasets


## 1. Validate Required Inputs


In [2]:
required_paths = [
    EARTHQUAKE_PATH,
    WEATHER_PATH,
    POPULATION_RASTER_PATH,
    POPULATION_SELECTED_METADATA_PATH,
]

missing_paths = [path for path in required_paths if not path.exists()]

if missing_paths:
    missing_text = "\n".join(str(path) for path in missing_paths)
    raise FileNotFoundError(
        "Missing required preprocessing inputs. Run the API notebooks first:\n"
        f"{missing_text}"
    )

with POPULATION_SELECTED_METADATA_PATH.open("r", encoding="utf-8") as file:
    population_metadata = json.load(file)

print("WorldPop dataset:", population_metadata.get("title"))
print("WorldPop ISO3:", population_metadata.get("iso3"))
print("WorldPop year:", population_metadata.get("popyear"))

if REQUIRE_EVENT_INFRASTRUCTURE and not EVENT_INFRASTRUCTURE_PATH.exists():
    raise FileNotFoundError(
        "event_infrastructure_summary.csv was not found. "
        "Run OSM_API.ipynb first so final_disaster_dataset.csv does not use stale or zero infrastructure counts."
    )


WorldPop dataset: The spatial distribution of population in 2020, India
WorldPop ISO3: IND
WorldPop year: 2020


## 2. Load and Clean Earthquake Data


In [3]:
earthquake_df = pd.read_csv(EARTHQUAKE_PATH)
if "SourceEventID" not in earthquake_df.columns:
    earthquake_df = earthquake_df.reset_index().rename(columns={"index": "SourceEventID"})

required_earthquake_columns = [

    "SourceEventID",

    "Magnitude",

    "Latitude",

    "Longitude",

    "Depth",

    "Time"

]

missing_columns = [
    column for column in required_earthquake_columns
    if column not in earthquake_df.columns
]

if missing_columns:
    raise ValueError(f"Earthquake data is missing columns: {missing_columns}")

earthquake_df = earthquake_df[required_earthquake_columns].copy()
earthquake_df["DisasterType"] = "Earthquake"

for column in ["Magnitude", "Latitude", "Longitude", "Depth"]:
    earthquake_df[column] = pd.to_numeric(earthquake_df[column], errors="coerce")

earthquake_df["Time"] = pd.to_datetime(
    earthquake_df["Time"],
    unit="ms",
    errors="coerce",
)

earthquake_df = earthquake_df.dropna(subset=required_earthquake_columns).reset_index(drop=True)

print("Earthquake rows:", len(earthquake_df))
earthquake_df.head()


Earthquake rows: 18702


,SourceEventID,Magnitude,Latitude,Longitude,Depth,Time,DisasterType
0,0,4.3,25.5695,90.7428,10.000,2026-07-12 06:30:39.013,Earthquake
1,1,4.5,16.4995,84.5232,10.000,2026-07-11 23:35:45.479,Earthquake
2,2,4.2,22.6970,95.8926,10.000,2026-07-11 22:39:24.089,Earthquake
3,3,4.2,19.2225,76.8290,10.000,2026-07-08 20:07:25.081,Earthquake
4,4,4.3,37.3926,72.1298,216.967,2026-07-08 06:14:54.818,Earthquake


Cell 5: Combine USGS + GDACS

In [4]:
# ===================================
# Combine USGS and GDACS disasters
# ===================================


# -----------------------
# USGS Earthquake data
# -----------------------

earthquake_df["DisasterType"] = "Earthquake"



# -----------------------
# Load GDACS disasters
# -----------------------

gdacs_df = pd.read_csv(
    GDACS_DATA_PATH
)



print(
    "GDACS Columns:"
)

print(
    gdacs_df.columns
)



# Rename GDACS columns to match USGS


gdacs_df = gdacs_df.rename(

    columns={

        "GDACSEventID":
        "SourceEventID",


        "Published":
        "Time"

    }

)
gdacs_mapping = {

    "EQ": "Earthquake",

    "FL": "Flood",

    "TC": "Cyclone",

    "DR": "Drought",

    "WF": "Wildfire",

    "VO": "Volcano"

}


gdacs_df["DisasterType"] = (

    gdacs_df["DisasterType"]
    .replace(gdacs_mapping)

)




# GDACS already has:
# DisasterType column
# Flood, Cyclone, etc.



# GDACS does not contain earthquake fields


gdacs_df["Magnitude"] = 0


gdacs_df["Depth"] = 0




# Common format for ML dataset


common_columns = [

    "SourceEventID",

    "Latitude",

    "Longitude",

    "Time",

    "Magnitude",

    "Depth",

    "DisasterType"

]





# Keep only required columns


earthquake_part = (

    earthquake_df[
        common_columns
    ]

)


gdacs_part = (

    gdacs_df[
        common_columns
    ]

)




# Combine


combined_df = pd.concat(

    [

        earthquake_part,

        gdacs_part

    ],

    ignore_index=True

)





# Save combined disaster file


combined_df.to_csv(

    COMBINED_DATA_PATH,

    index=False

)




# Continue preprocessing with combined data


earthquake_df = (

    combined_df.copy()

)





print(
    "Combined disaster count:"
)


print(

    earthquake_df["DisasterType"]
    .value_counts()

)



combined_df.head()

GDACS Columns:
Index(['GDACSEventID', 'DisasterType', 'AlertLevel', 'Severity', 'Country',
       'Latitude', 'Longitude', 'Title', 'Published', 'Link', 'Summary'],
      dtype='object')
Combined disaster count:
DisasterType
Earthquake    18706
Wildfire        185
Flood            12
Drought          10
Volcano           1
Cyclone           1
Name: count, dtype: int64


,SourceEventID,Latitude,Longitude,Time,Magnitude,Depth,DisasterType
0,0,25.5695,90.7428,2026-07-12 06:30:39.013000,4.3,10.000,Earthquake
1,1,16.4995,84.5232,2026-07-11 23:35:45.479000,4.5,10.000,Earthquake
2,2,22.6970,95.8926,2026-07-11 22:39:24.089000,4.2,10.000,Earthquake
3,3,19.2225,76.8290,2026-07-08 20:07:25.081000,4.2,10.000,Earthquake
4,4,37.3926,72.1298,2026-07-08 06:14:54.818000,4.3,216.967,Earthquake


In [5]:
# =====================================
# Load and Merge Weather Data
# =====================================


weather_df = pd.read_csv(
    WEATHER_DATA_PATH
)


print(
    "Weather rows:",
    len(weather_df)
)


print(
    weather_df.columns
)



required_weather_columns = [

    "Temperature",

    "Humidity",

    "Rainfall",

    "WindSpeed"

]



weather_columns = [

    "SourceEventID"

] + required_weather_columns



weather_df = (

    weather_df[
        weather_columns
    ]

)



earthquake_df = earthquake_df.merge(

    weather_df,

    on="SourceEventID",

    how="left"

)



for column in required_weather_columns:

    earthquake_df[column] = (

        earthquake_df[column]
        .fillna(
            earthquake_df[column].mean()
        )

    )



print(
    earthquake_df[
        required_weather_columns
    ].head()
)

Weather rows: 2351
Index(['SourceEventID', 'Latitude', 'Longitude', 'EventTime', 'WeatherTime',
       'Temperature', 'Humidity', 'Rainfall', 'WindSpeed'],
      dtype='object')
   Temperature   Humidity  Rainfall  WindSpeed
0    18.419779  72.307954  0.188473    9.40923
1    18.419779  72.307954  0.188473    9.40923
2    18.419779  72.307954  0.188473    9.40923
3    18.419779  72.307954  0.188473    9.40923
4    18.419779  72.307954  0.188473    9.40923


In [6]:
earthquake_df.shape

(18915, 11)

## 3. Load and Summarize Weather Data


In [7]:
weather_df = pd.read_csv(WEATHER_PATH)

weather_column_map = {
    "temperature": "Temperature",
    "humidity": "Humidity",
    "rainfall": "Rainfall",
    "wind_speed": "WindSpeed",
}

weather_df = weather_df.rename(columns=weather_column_map)

required_weather_columns = ["Temperature", "Humidity", "Rainfall", "WindSpeed"]
missing_weather_columns = [
    column for column in required_weather_columns
    if column not in weather_df.columns
]

if missing_weather_columns:
    raise ValueError(f"Weather data is missing columns: {missing_weather_columns}")

for column in required_weather_columns:
    weather_df[column] = pd.to_numeric(weather_df[column], errors="coerce")

weather_event_df = None
weather_summary = None

if "SourceEventID" in weather_df.columns:
    weather_event_df = weather_df[["SourceEventID", *required_weather_columns]].copy()
    weather_event_df = weather_event_df.drop_duplicates(subset=["SourceEventID"], keep="last")
    weather_source = "event-local historical Open-Meteo"
else:
    weather_summary = weather_df[required_weather_columns].mean(numeric_only=True)

    if weather_summary.isna().any():
        raise ValueError("Weather summary contains missing values after numeric conversion.")

    weather_source = "global mean weather fallback"
    print(
        "Warning: weather_data.csv has no SourceEventID column. "
        "Using one mean weather value for every event. Run OpenMeteo_API.ipynb again for event-local weather."
    )

print(f"Weather source: {weather_source}")
weather_df.head()


Weather source: event-local historical Open-Meteo


,SourceEventID,Latitude,Longitude,EventTime,WeatherTime,Temperature,Humidity,Rainfall,WindSpeed
0,174,34.5055,74.1714,2024-12-27 15:37:04.335,2024-12-27 16:00:00,0.6,68.0,0.0,0.2
1,1322,24.6737,93.5219,2024-11-27 12:50:07.302,2024-11-27 13:00:00,17.4,72.0,0.0,5.3
2,1716,23.6308,72.0385,2024-11-15 16:45:36.582,2024-11-15 17:00:00,23.1,51.0,0.0,2.5
3,6185,34.1455,74.3501,2024-07-12 06:56:22.699,2024-07-12 07:00:00,28.9,33.0,0.0,6.8
4,6464,24.3726,92.6936,2024-07-04 21:12:27.022,2024-07-04 21:00:00,25.3,99.0,0.1,3.3


## 4. Add India WorldPop Population Values


In [8]:
def point_inside_bounds(latitude, longitude, bounds):
    return (
        bounds.left <= longitude <= bounds.right
        and bounds.bottom <= latitude <= bounds.top
    )


def get_population_from_raster(dataset, latitude, longitude):
    if not point_inside_bounds(latitude, longitude, dataset.bounds):
        return 0.0, False

    row, col = dataset.index(longitude, latitude)

    if row < 0 or col < 0 or row >= dataset.height or col >= dataset.width:
        return 0.0, False

    value = float(dataset.read(1, window=Window(col, row, 1, 1))[0, 0])

    if dataset.nodata is not None and value == dataset.nodata:
        return 0.0, False
    if value < 0 or np.isnan(value):
        return 0.0, False

    return value, True


def get_population_sum_within_radius(dataset, latitude, longitude, radius_km=25):
    if not point_inside_bounds(latitude, longitude, dataset.bounds):
        return 0.0

    # Compute bounding window for the radius
    lat_radius = radius_km / 111.32
    lon_radius = radius_km / (111.32 * max(np.cos(np.radians(latitude)), 0.01))
    left = max(dataset.bounds.left, longitude - lon_radius)
    right = min(dataset.bounds.right, longitude + lon_radius)
    bottom = max(dataset.bounds.bottom, latitude - lat_radius)
    top = min(dataset.bounds.top, latitude + lat_radius)
    window = rasterio.windows.from_bounds(left, bottom, right, top, dataset.transform)
    window = window.round_offsets().round_lengths()

    # Read the data in the window
    data = dataset.read(1, window=window, boundless=False).astype(float)
    if data.size == 0:
        return 0.0

    # Compute geographic coordinates of each pixel center
    rows, cols = np.indices(data.shape)
    row_off = int(window.row_off)
    col_off = int(window.col_off)
    xs, ys = rasterio.transform.xy(dataset.transform, rows + row_off, cols + col_off, offset="center")
    xs = np.asarray(xs)
    ys = np.asarray(ys)

    # Calculate distances from the epicenter
    km_per_lon_degree = 111.32 * max(np.cos(np.radians(latitude)), 0.01)
    distances_km = np.sqrt(((ys - latitude) * 111.32) ** 2 + ((xs - longitude) * km_per_lon_degree) ** 2)

    # Flatten arrays to avoid broadcasting issues
    flat_data = data.flatten()
    flat_dist = distances_km.flatten()

    # Apply masks
    valid = (flat_dist <= radius_km)
    if dataset.nodata is not None:
        valid &= (flat_data != dataset.nodata)
    valid &= np.isfinite(flat_data) & (flat_data >= 0)

    return float(np.sum(flat_data[valid]))

population_pixel_values = []
population_radius_values = []
inside_population_values = []


with rasterio.open(POPULATION_RASTER_PATH) as population_raster:

    print(
        "Population raster CRS:",
        population_raster.crs
    )

    print(
        "Population raster bounds:",
        population_raster.bounds
    )


    for row in earthquake_df.itertuples(index=False):

        population_pixel, inside = (
            get_population_from_raster(
                population_raster,
                row.Latitude,
                row.Longitude,
            )
        )


        population_radius = (
    get_population_sum_within_radius(

        population_raster,

        row.Latitude,

        row.Longitude,

        DEFAULT_INFRASTRUCTURE_RADIUS_KM

    )
)


        population_pixel_values.append(
            population_pixel
        )

        population_radius_values.append(
            population_radius
        )

        inside_population_values.append(
            inside
        )



earthquake_df["PopulationAtEpicenterPixel"] = (
    population_pixel_values
)


earthquake_df["Population"] = (
    population_radius_values
)


earthquake_df["InsidePopulationRaster"] = (
    inside_population_values
)


earthquake_df["PopulationDataset"] = (
    "WorldPop"
)


earthquake_df[
    [
        "Latitude",
        "Longitude",
        "Population",
        "PopulationAtEpicenterPixel"
    ]
].head()


Population raster CRS: EPSG:4326
Population raster bounds: BoundingBox(left=68.19583234055, bottom=6.755833642309984, right=97.39583222375, top=35.51166686061998)


,Latitude,Longitude,Population,PopulationAtEpicenterPixel
0,25.5695,90.7428,270097.484934,0.0
1,16.4995,84.5232,0.000000,0.0
2,22.6970,95.8926,0.000000,0.0
3,19.2225,76.8290,748506.467547,0.0
4,37.3926,72.1298,0.000000,0.0


In [9]:
earthquake_df["Population"].describe()

count    1.891500e+04
mean     3.868113e+04
std      4.674690e+05
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      0.000000e+00
max      2.649584e+07
Name: Population, dtype: float64

In [10]:
(
    earthquake_df["Population"] > 0
).sum()

np.int64(2379)

In [11]:
with rasterio.open(
    POPULATION_RASTER_PATH
) as src:

    data = src.read(1)

    print(
        "Min:",
        np.nanmin(data)
    )

    print(
        "Max:",
        np.nanmax(data)
    )

    print(
        "Non zero:",
        np.sum(data > 0)
    )

Min: -99999.0
Max: 1918.7952
Non zero: 140086378


In [12]:
# Remove disasters with no exposed population

before = len(
    earthquake_df
)


earthquake_df = earthquake_df[

    earthquake_df["Population"] > 0

].copy()


after = len(
    earthquake_df
)


print(
    "Removed:",
    before-after
)


print(
    "Remaining useful disaster events:",
    after
)

Removed: 16536
Remaining useful disaster events: 2379


In [13]:
USEFUL_EVENTS_PATH = (
    DATA_DIR / "useful_disaster_events.csv"
)


earthquake_df.to_csv(

    USEFUL_EVENTS_PATH,

    index=False

)


print(
    "Useful events saved:",
    len(earthquake_df)
)

Useful events saved: 2379


In [14]:
IMPACT_RADIUS_KM = 25


AREA_KM2 = (
    np.pi
    *
    IMPACT_RADIUS_KM ** 2
)


earthquake_df["PopulationDensity"] = (

    earthquake_df["Population"]

    /

    AREA_KM2

)

## 6. Severity and Impact Score


In [15]:
def classify_severity(magnitude):

    if magnitude >= 7:
        return "High"

    elif magnitude >= 5:
        return "Medium"

    else:
        return "Low"



earthquake_df["Severity"] = (

    earthquake_df["Magnitude"]
    .apply(classify_severity)

)



earthquake_df["ImpactScore"] = (


    earthquake_df["Magnitude"] * 10


    + np.log1p(

        earthquake_df["PopulationDensity"]

    ) * 3


    + earthquake_df["Rainfall"] * 0.5


    + earthquake_df["WindSpeed"] * 0.2


)



earthquake_df["ImpactScore"] = (

    earthquake_df["ImpactScore"]
    .clip(lower=0)

)

## 7. Build Final Dataset


In [16]:
final_columns = [

    "SourceEventID",

    "DisasterType",

    "Latitude",

    "Longitude",

    "Magnitude",

    "Depth",

    "Time",


    "Temperature",

    "Humidity",

    "Rainfall",

    "WindSpeed",


    "Population",

    "PopulationDensity",


    "Severity",

    "ImpactScore"

]


final_df = earthquake_df[
    final_columns
].copy()


print(final_df.shape)

final_df.head()

(2379, 15)


,SourceEventID,DisasterType,Latitude,Longitude,Magnitude,Depth,Time,Temperature,Humidity,Rainfall,WindSpeed,Population,PopulationDensity,Severity,ImpactScore
0,0,Earthquake,25.5695,90.7428,4.3,10.0,2026-07-12 06:30:39.013000,18.419779,72.307954,0.188473,9.40923,270097.484934,137.559520,Low,59.769982
3,3,Earthquake,19.2225,76.8290,4.2,10.0,2026-07-08 20:07:25.081000,18.419779,72.307954,0.188473,9.40923,748506.467547,381.211214,Low,61.814003
26,26,Earthquake,27.4893,92.2066,4.2,10.0,2026-06-21 02:24:55.135000,18.419779,72.307954,0.188473,9.40923,15255.286687,7.769454,Low,50.489906
28,28,Earthquake,27.5051,92.1524,4.3,10.0,2026-06-21 00:53:20.927000,18.419779,72.307954,0.188473,9.40923,18414.441369,9.378398,Low,51.995262
29,29,Earthquake,27.4598,92.0424,4.3,10.0,2026-06-21 00:52:54.610000,18.419779,72.307954,0.188473,9.40923,29143.226981,14.842524,Low,53.264176


In [17]:
null_counts = final_df.isnull().sum()

if null_counts.any():
    raise ValueError(
        "Final dataset contains null values:\n"
        + null_counts[null_counts > 0].to_string()
    )

final_df.describe(include="all").transpose()


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
SourceEventID,2379.0,NaN,NaN,NaN,10872.013871,23089.60826,0.0,5428.0,11257.0,15332.5,1103972.0
DisasterType,2379,2,Earthquake,2378,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Latitude,2379.0,NaN,NaN,NaN,22.685428,8.639812,6.757,13.4875,24.511,29.0804,35.479
Longitude,2379.0,NaN,NaN,NaN,86.45864,8.85456,68.963,76.7701,92.4488,93.221,97.3521
Magnitude,2379.0,NaN,NaN,NaN,4.408008,0.460838,0.0,4.1,4.4,4.6,7.7
Depth,2379.0,NaN,NaN,NaN,30.643967,20.44923,0.0,10.0,30.0,38.5,131.4
Time,2379,2379,"Thu, 25 Jun 2026 07:47:24 GMT",1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Temperature,2379.0,NaN,NaN,NaN,18.464716,4.833675,-27.4,18.419779,18.419779,18.419779,35.6
Humidity,2379.0,NaN,NaN,NaN,71.18572,11.255936,5.0,72.307954,72.307954,72.307954,100.0
Rainfall,2379.0,NaN,NaN,NaN,0.190148,0.277229,0.0,0.188473,0.188473,0.188473,7.0


## 8. Save Final Dataset


In [18]:
final_df.to_csv(FINAL_DATASET_PATH, index=False)

print(f"Final dataset saved: {FINAL_DATASET_PATH}")
print(f"Rows saved: {len(final_df)}")


Final dataset saved: C:\Users\aashi\PycharmProjects\AI-Based Disaster Response Management System for Resource Allocation and Relief Coordination\datasets\final_disaster_dataset.csv
Rows saved: 2379


In [19]:
print("=" * 48)
print("Data preprocessing completed successfully")
print(f"Scope: India WorldPop-covered disaster events")
print(f"WorldPop source: {population_metadata.get('iso3')} {population_metadata.get('popyear')}")
print(f"Weather source: {weather_source}")
print(f"Infrastructure source: event-local OSM within {DEFAULT_INFRASTRUCTURE_RADIUS_KM:.1f} km unless summary says otherwise")
print(f"Output: {FINAL_DATASET_PATH}")
print("=" * 48)


Data preprocessing completed successfully
Scope: India WorldPop-covered disaster events
WorldPop source: IND 2020
Weather source: event-local historical Open-Meteo
Infrastructure source: event-local OSM within 25.0 km unless summary says otherwise
Output: C:\Users\aashi\PycharmProjects\AI-Based Disaster Response Management System for Resource Allocation and Relief Coordination\datasets\final_disaster_dataset.csv
